In [1]:
!pip install -q transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

print("All imports successful")

All imports successful


In [3]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

HfUriError: Invalid HF URI 'hf://datasets/imdb@e6281661ce1c48d982bc483cf8a173c1bbeb5d31/.huggingface.yaml'. Repository id must be 'namespace/name', got 'imdb'.

In [4]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize,
    batched=True
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [6]:
tokenized_dataset["train"][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
small_train = tokenized_dataset["train"].shuffle(seed=42).select(range(2000))

small_test = tokenized_dataset["test"].shuffle(seed=42).select(range(500))

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1
)

In [15]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [16]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.370077,0.842000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.47524008178710936, metrics={'train_runtime': 3035.1804, 'train_samples_per_second': 0.659, 'train_steps_per_second': 0.082, 'total_flos': 131555527680000.0, 'train_loss': 0.47524008178710936, 'epoch': 1.0})

### Notebook Explanation: Fine-tuning a Sentiment Analysis Model

This notebook demonstrates how to fine-tune a pre-trained BERT model for sentiment analysis using the Hugging Face Transformers and Datasets libraries. We'll cover everything from installing necessary libraries to training the model.


#### Cell `qqUjGhmEduYQ`: Install Libraries

```python
!pip install -q transformers datasets evaluate accelerate
```

This cell installs the required Python libraries using `pip`. The `-q` flag stands for 'quiet', meaning it suppresses most of the output during installation.

*   `transformers`: This library provides state-of-the-art pre-trained models like BERT for various NLP tasks.
*   `datasets`: This library makes it easy to load and preprocess datasets for machine learning.
*   `evaluate`: This library provides a unified API for various evaluation metrics.
*   `accelerate`: This library helps to train models faster, especially on multiple GPUs or TPUs, by simplifying distributed training.


#### Cell `vCAJ7cA1d_4d`: Import Core Components

```python
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

print("All imports successful")
```

Here, we import specific functions and classes from the newly installed libraries:

*   `load_dataset` (from `datasets`): Used to easily download and load datasets from the Hugging Face Hub.
*   `AutoTokenizer` (from `transformers`): A class that automatically loads the correct tokenizer for a given pre-trained model. A tokenizer converts text into numerical IDs that a model can understand.
*   `AutoModelForSequenceClassification` (from `transformers`): A class that automatically loads a pre-trained model suitable for sequence classification tasks (like sentiment analysis).
*   `Trainer` (from `transformers`): A high-level API for training and evaluating models, simplifying the training loop.
*   `TrainingArguments` (from `transformers`): A class to define all the hyperparameters and settings for training.

The `print` statement simply confirms that all the necessary imports were successful.


#### Cells `mc2DnMTqeU6y` and `b5EBNb9Nek52`: Load the IMDB Dataset

Initially, an attempt was made to load the IMDB dataset using `dataset = load_dataset("imdb")` (Cell `mc2DnMTqeU6y`). This resulted in an `HfUriError` because the `datasets` library, specifically the `huggingface_hub` component, expected the dataset identifier in the format `namespace/name` (e.g., `owner/dataset_name`). While `imdb` is a well-known dataset, sometimes specifying its full path, `stanfordnlp/imdb`, resolves such naming ambiguities.

```python
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

print(dataset)
```

In Cell `b5EBNb9Nek52`, the dataset is successfully loaded by specifying the full path `"stanfordnlp/imdb"`. The `imdb` dataset is a popular dataset for binary sentiment classification, containing movie reviews labeled as positive (1) or negative (0). The `print(dataset)` command shows the structure of the loaded dataset, which typically includes 'train' and 'test' splits.


#### Cell `1JAVSHZweoCp`: Tokenize the Dataset

```python
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize,
    batched=True
)
```

This cell performs tokenization, a crucial step in NLP where text is converted into numerical representations that a neural network can process.

1.  **`tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")`**: We load a tokenizer that corresponds to the `bert-base-uncased` model. This ensures that the text is preprocessed in the same way the BERT model was pre-trained (e.g., splitting words, adding special tokens, converting to lowercase).

2.  **`tokenize(example)` function**: This function takes an `example` (which is a row from the dataset containing a 'text' field) and applies the tokenizer:
    *   `example["text"]`: The input text from the dataset.
    *   `truncation=True`: If a text sequence is longer than `max_length`, it will be cut short.
    *   `padding="max_length"`: If a text sequence is shorter than `max_length`, it will be padded with special tokens to reach `max_length`.
    *   `max_length=128`: All input sequences will be processed to a fixed length of 128 tokens. This is important for batch processing in neural networks.

3.  **`tokenized_dataset = dataset.map(tokenize, batched=True)`**: The `map` method applies the `tokenize` function to all examples in the `dataset`. `batched=True` means the tokenizer processes multiple examples at once, which is more efficient.


#### Cell `bsYq-voDfjem`: Inspect Tokenized Data

```python
tokenized_dataset["train"][0]
```

This cell displays the first example from the `train` split of our `tokenized_dataset`. After tokenization, each example now contains:

*   `text`: The original movie review text.
*   `label`: The sentiment label (0 for negative, 1 for positive).
*   `input_ids`: The numerical IDs corresponding to each token in the text.
*   `token_type_ids`: Identifies which sequence a token belongs to (useful for tasks with two input sequences, like question answering).
*   `attention_mask`: Indicates which tokens are actual content (1) and which are padding (0), so the model doesn't pay attention to padding tokens.


#### Cell `Bb_UhSygfmXO`: Load Pre-trained Model for Sequence Classification

```python
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)
```

This cell loads a pre-trained BERT model adapted for sequence classification:

*   **`AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)`**: We load the `bert-base-uncased` model. `"uncased"` means it was trained without distinguishing between uppercase and lowercase letters. `num_labels=2` tells the model that it needs to classify inputs into two categories (positive/negative sentiment in our case).

*   **Warnings (`UNEXPECTED`, `MISSING`)**: When you run this, you might see warnings about `UNEXPECTED` and `MISSING` keys. These are common and generally safe to ignore in transfer learning scenarios:
    *   `UNEXPECTED`: The pre-trained `bert-base-uncased` model has a head (like `cls.predictions`) that was used for its original pre-training task (e.g., masked language modeling, next sentence prediction). When we load it for sequence classification, these original heads are replaced by a new classification head, so the old weights are 'unexpected' for the new architecture.
    *   `MISSING`: The new `classifier.weight` and `classifier.bias` parameters for our `num_labels=2` classification task are newly initialized. They were `MISSING` from the original pre-trained model because it didn't have a sentiment classification head. These new parameters will be learned during our fine-tuning process.


#### Cell `EFKJC-wyfsUT`: Create Smaller Subsets for Training and Evaluation

```python
small_train = tokenized_dataset["train"].shuffle(seed=42).select(range(2000))

small_test = tokenized_dataset["test"].shuffle(seed=42).select(range(500))
```

Training a large model on a full dataset can be computationally expensive and time-consuming. This cell creates smaller subsets of the training and testing data for quicker experimentation and demonstration.

*   **`.shuffle(seed=42)`**: Randomly shuffles the dataset. Using a `seed` (e.g., 42) ensures that the shuffling is reproducible, meaning you get the same random order every time you run the code.
*   **`.select(range(2000))`**: Selects the first 2000 examples from the shuffled training dataset to create `small_train`.
*   **`.select(range(500))`**: Selects the first 500 examples from the shuffled test dataset to create `small_test`.


#### Cell `5FVwWXn4fwWs`: Define Training Arguments

```python
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1
)
```

`TrainingArguments` is where you define all the hyperparameters and configurations for your training process. Here's a breakdown of the key arguments used:

*   **`output_dir="./results"`**: Specifies the directory where the model checkpoints and training logs will be saved.
*   **`eval_strategy="epoch"`**: The evaluation will be performed at the end of each epoch (a full pass over the training data).
*   **`save_strategy="epoch"`**: The model will be saved at the end of each epoch.
*   **`learning_rate=2e-5`**: This is the step size at which the model's weights are updated during training. A common learning rate for fine-tuning pre-trained transformers is a small value like `2e-5` (0.00002).
*   **`per_device_train_batch_size=8`**: The number of training examples processed in one go on each device (GPU/CPU). Smaller batch sizes require less memory but might lead to noisier updates.
*   **`per_device_eval_batch_size=8`**: Similar to `train_batch_size`, but for evaluation.
*   **`num_train_epochs=1`**: The total number of times the model will iterate over the entire training dataset. For fine-tuning, often 1 to 3 epochs are sufficient due to the pre-trained nature of the model.


#### Cell `1GGprIxofz-f`: Define Metrics Computation

```python
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )
```

This cell sets up how the model's performance will be evaluated during training.

1.  **`import evaluate`**: Imports the `evaluate` library, which provides easy access to various evaluation metrics.
2.  **`accuracy = evaluate.load("accuracy")`**: Loads the accuracy metric from the `evaluate` library. This will be used to calculate the proportion of correctly classified examples.
3.  **`compute_metrics(eval_pred)` function**: This function takes the model's raw outputs (`logits`) and the true labels (`labels`) from the evaluation set.
    *   `logits`: These are the raw, unnormalized scores output by the model for each class. For binary classification (2 labels), `logits` will be a tensor with two values for each example.
    *   `predictions = np.argmax(logits, axis=-1)`: `np.argmax` finds the index of the highest logit value for each example, which corresponds to the model's predicted class (0 or 1).
    *   `return accuracy.compute(...)`: The calculated `predictions` and the true `labels` are then passed to the `accuracy.compute` function, which returns the accuracy score.


#### Cell `axS8KUchf6Jw`: Initialize the Trainer

```python
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
    compute_metrics=compute_metrics
)
```

The `Trainer` class from the `transformers` library orchestrates the entire training and evaluation process. This cell initializes the `Trainer` by bringing together all the components we defined earlier:

*   **`model=model`**: The pre-trained `AutoModelForSequenceClassification` we loaded.
*   **`args=training_args`**: The `TrainingArguments` object defining all the training parameters.
*   **`train_dataset=small_train`**: The subset of our tokenized IMDB training data.
*   **`eval_dataset=small_test`**: The subset of our tokenized IMDB test data, used for evaluating the model's performance during training.
*   **`compute_metrics=compute_metrics`**: The function we defined to calculate evaluation metrics (in our case, accuracy).

By encapsulating all these elements, the `Trainer` simplifies the training loop, handling aspects like optimization, learning rate scheduling, logging, and saving checkpoints.


#### Cell `oEcPu1q_f8w3`: Start Training

```python
trainer.train()
```

This is the final step that kicks off the model training process. When you execute `trainer.train()`:

*   The model will iterate over the `small_train` dataset for the number of `num_train_epochs` specified in `training_args`.
*   During each epoch, it will perform forward and backward passes, update model weights using the specified optimizer and learning rate.
*   At the end of each epoch (as per `eval_strategy="epoch"`), it will evaluate the model on the `small_test` dataset using the `compute_metrics` function.
*   It will save the model checkpoint at the end of each epoch (as per `save_strategy="epoch"`).
*   It will display training progress and evaluation results in the output.

After this cell completes, your BERT model will have been fine-tuned for sentiment analysis on the IMDB dataset!


In [18]:
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
No log,0.370077,1,0.842000


{'eval_loss': 0.3700771927833557, 'eval_accuracy': 0.842}

In [19]:
model.save_pretrained("./sentiment_model")
tokenizer.save_pretrained("./sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./sentiment_model/tokenizer_config.json', './sentiment_model/tokenizer.json')

In [21]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer
)